# Test ELT Process

## Environment Setup & Mock Data Generation

In [ ]:
# --- JUPYTER MAGIC ---
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
import pandas as pd
from pathlib import Path

# Add project root to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.config import load_settings, root_path
from src.state.schema import init_schema, get_unified_staging_view
from src.elt.scanner import scan_sources
from src.elt.strategy import generate_strategy
from src.elt.ingestion import execute_ingestion
from prefect.blocks.system import Secret
from openai import OpenAI

# Create Mock Directories for Testing
mock_gdrive = root_path("notebooks/mock_gdrive")
mock_onedrive = root_path("notebooks/mock_onedrive")
mock_nextcloud = root_path("notebooks/mock_nextcloud")

for d in [mock_gdrive, mock_onedrive, mock_nextcloud]:
    os.makedirs(d, exist_ok=True)

# Generate Mock Files (Notice 'tax_2026.pdf' is exactly the same in both drives!)
(mock_gdrive / "tax_2026.pdf").write_text("DUMMY PDF CONTENT: TAX 2026")
(mock_gdrive / "aws_invoice_july.txt").write_text("AWS BILL: $150")

(mock_onedrive / "tax_2026.pdf").write_text("DUMMY PDF CONTENT: TAX 2026") # EXACT DUPLICATE
(mock_onedrive / "family_photo.jpg").write_text("DUMMY JPEG CONTENT: BEACH")

print("✅ Mock Cloud Drives and Files Generated.")

## Phase 1 - Multi-Source Staging

In [ ]:
# Define our sources
cloud_sources = {
    "Google Drive": str(mock_gdrive),
    "OneDrive": str(mock_onedrive)
}

# Use a test database so we don't mess up your real ledger
test_db = root_path("data/test_ledger.db")

print("--- PHASE 1: MULTI-SOURCE SCANNING ---")
init_schema(test_db, list(cloud_sources.keys()))
scan_sources(test_db, cloud_sources)

# Verify Staging Isolation
conn = sqlite3.connect(test_db)
print("\n📊 Staging Table: Google Drive")
display(pd.read_sql_query("SELECT original_path, sha256_hash, status FROM staging_google_drive", conn))

print("\n📊 Staging Table: OneDrive")
display(pd.read_sql_query("SELECT original_path, sha256_hash, status FROM staging_onedrive", conn))
conn.close()

## Phase 2 - Cross-Table Dedupe & Agentic Strategy

In [ ]:
print("--- PHASE 2: CROSS-TABLE DEDUPLICATION & STRATEGY ---")

# 1. Test SQL Deduplication
unified_view = get_unified_staging_view(test_db, list(cloud_sources.keys()))
print(f"\nUnique Files to Route: {len(unified_view)} (Expected 3, because the tax PDF is a duplicate!)")
display(pd.DataFrame(unified_view))

# 2. Test Agentic Routing
print("\n🧠 Authenticating with Prefect Vault...")
llm_key = await Secret.load("nas-gemini-api-key")
llm_client = OpenAI(api_key=llm_key.get(), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

print("🤖 Asking Agent to generate taxonomy strategy...")
routings = generate_strategy(test_db, list(cloud_sources.keys()), llm_client, "gemini-2.5-pro")

print("\n🎯 Agentic Routing Strategy:")
for r in routings:
    print(f" - [Staging ID {r.staging_id}] -> {r.proposed_path}")
    print(f"   Reasoning: {r.reasoning}\n")

## Phase 3 - Physical Ingestion & Production State

In [ ]:
print("--- PHASE 3: PHYSICAL INGESTION ---")

# Execute the actual v5 ingestion engine!
# It natively uses shutil.copy2 to move files to our mock_nextcloud ZFS mount
execute_ingestion(
    db_path=test_db, 
    nextcloud_mount=mock_nextcloud, 
    routings=routings
)

# Verify the Production Inventory Source of Truth
conn = sqlite3.connect(test_db)
print("\n📊 Final Production Inventory:")
display(pd.read_sql_query("SELECT sha256_hash, nextcloud_path, file_size FROM production_inventory", conn))

# Verify Staging Tables were marked as 'ingested' or 'duplicate'
print("\n🔍 Final Google Drive Staging State:")
display(pd.read_sql_query("SELECT original_path, status FROM staging_google_drive", conn))

print("\n🔍 Final OneDrive Staging State:")
display(pd.read_sql_query("SELECT original_path, status FROM staging_onedrive", conn))

conn.close()